In [1]:
!pip install -q duckdb

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



In [4]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [5]:
columns_df = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 1
""").df()

print(columns_df[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## Unit of Analysis + Time Window

- **One row:** One content page (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).
- **Table:** `fact_content_daily_performance`
- **Time window:** A single mid-panel month (`month = '2026-03'`) to avoid using the final month as instructed.
- **Prediction / ranking objective:** Rank content pages for manual review using search and engagement signals as a decision-support task.
- **Excluded:** Client identifiers (`client_hash_id`) are excluded as model features because they identify the source rather than page performance and would not generalize to unseen clients.

In [6]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS pages,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────┬────────┬────────────┬────────────┐
│  rows   │ clients │ pages  │ start_date │  end_date  │
│  int64  │  int64  │ int64  │    date    │    date    │
├─────────┼─────────┼────────┼────────────┼────────────┤
│ 9841378 │      55 │ 331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────┴────────┴────────────┴────────────┘



## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

### Label / Proxy
No final label exists in this warehouse table. For this notebook, the objective is to prepare features that could later be used for ranking or predicting content requiring review.

### Context
- report_date
- month

### Excluded
- client_hash_id
- content_hash_id

These identifiers uniquely identify clients and pages but should not be used as predictive features because they do not represent page performance and would reduce generalization.

## Verification Queries

The following queries verify the data contract using a single mid-panel month (2026-03).

1. Verify the grain and reporting window.
2. Verify the number of rows and unique content pages.
3. Verify data availability using `IS TRUE`.

In [7]:
# Query 1: Verify grain and reporting window
print("Query 1: Grain and reporting window")

con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS pages,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").show()


# Query 2: Verify one row per client-page-date combination
print("\nQuery 2: Grain verification")

con.sql(f"""
SELECT
COUNT(*) AS duplicate_keys
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month='2026-03'
    GROUP BY 1,2,3
    HAVING COUNT(*)>1
)
""").show()


# Query 3: Availability check
print("\nQuery 3: Availability")

con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE
""").show()

Query 1: Grain and reporting window
┌─────────┬─────────┬────────┬────────────┬────────────┐
│  rows   │ clients │ pages  │ start_date │  end_date  │
│  int64  │  int64  │ int64  │    date    │    date    │
├─────────┼─────────┼────────┼────────────┼────────────┤
│ 9841378 │      55 │ 331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────┴────────┴────────────┴────────────┘


Query 2: Grain verification


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ duplicate_keys │
│     int64      │
├────────────────┤
│              0 │
└────────────────┘


Query 3: Availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         364347 │
└────────────────┘



## Data Limits

This warehouse contains daily search and analytics observations but does not provide a final target label indicating whether a page should be refreshed. Any prediction target would need to be derived separately. The analysis is also limited to one month (2026-03), so longer-term trends and seasonality are not captured. Finally, this dataset is intended for decision-support and should not be interpreted as establishing causal relationships.